# Session 17 · Random Forest I

Session 16 left a problem: a single deep tree **memorises** and gets brittle. Today's
fix is simple — **don't trust one tree; grow a hundred slightly different ones and let
them vote.** That crowd is a **random forest**.

> ✏️ = your cell. Gaps never block the run.

## Step 1 · Setup — the fraud problem, same yardstick as Module 3

We reuse the imbalanced fraud data and judge with **precision** and **recall** (S12–13).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, accuracy_score)

fraud = pd.read_csv("../../../datasets/secondary/fraud_transactions.csv")
features = ["amount","hour_of_day","is_online","distance_from_home_km","transactions_last_24h"]
X, y = fraud[features], fraud['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('test transactions:', len(y_test), ' of which fraud:', int(y_test.sum()))

## Step 2 · Meet the crowd — 7 experts who each saw different evidence

A forest makes its trees differ two ways: each trains on a **random sample of rows**
(bootstrap) and each split may only look at a **random subset of features**. Let's build
7 such trees by hand and check: do they *disagree*?

In [ ]:
rng = np.random.default_rng(0)
preds = []
print('7 individual trees (random rows + random features each):')
for i in range(7):
    idx = rng.integers(0, len(X_train), len(X_train))          # bootstrap sample of rows
    tree = DecisionTreeClassifier(max_features='sqrt', random_state=i)  # random features/split
    tree.fit(X_train.iloc[idx], y_train.iloc[idx])
    p = tree.predict(X_test)
    preds.append(p)
    print(f'   tree {i}:  accuracy {accuracy_score(y_test, p):.3f}')

preds = np.array(preds)
disagree = ((preds.sum(0) > 0) & (preds.sum(0) < 7)).sum()
print()
print(f'These 7 trees DISAGREE on {disagree} of {len(y_test)} test transactions.')
print('Each is a slightly different expert. That difference is the whole point.')

## Step 3 · Take the vote — and meet sklearn's forest

The majority answer of the crowd is the forest's prediction. sklearn wraps all of this
in `RandomForestClassifier` — 100 trees by default.

In [ ]:
# our hand-rolled majority vote of the 7 trees
hand_vote = (preds.mean(axis=0) >= 0.5).astype(int)
print('majority vote of our 7 trees: accuracy', round(accuracy_score(y_test, hand_vote), 3))

# the real thing: 100 trees
forest = RandomForestClassifier(n_estimators=100, random_state=0)
forest.fit(X_train, y_train)
print('RandomForestClassifier(100):  accuracy', round(forest.score(X_test, y_test), 3))

## Step 4 · Measure the edge — forest vs a single tree

Fit one ordinary tree and compare confusion matrices on the fraud stakes. Watch the
**false alarms**.

In [ ]:
def draw_matrix(y_true, pred, ax, title):
    cm = confusion_matrix(y_true, pred, labels=[1, 0])   # fraud row first
    disp = ConfusionMatrixDisplay(cm, display_labels=['fraud', 'legit'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)
    tp, fn, fp, tn = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    ax.set_xlabel(f'predicted\ncaught={tp}  missed={fn}  false alarms={fp}')

In [ ]:
single = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
pred_tree   = single.predict(X_test)
pred_forest = forest.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
draw_matrix(y_test, pred_tree,   axes[0], 'Single decision tree')
draw_matrix(y_test, pred_forest, axes[1], 'Random forest (100 trees)')
plt.tight_layout(); plt.show()

for name, p in [('single tree', pred_tree), ('forest', pred_forest)]:
    print(f'{name:12s}  recall {recall_score(y_test, p):.3f}   precision {precision_score(y_test, p):.3f}')
print()
print('Same 11 frauds caught (recall unchanged), but false alarms drop 5 -> 3:')
print('precision 0.688 -> 0.786. Two honest customers spared a wrongly frozen card.')

## Step 5 · How many experts do we need?

Sweep the number of trees and watch accuracy climb, then plateau — more experts help,
with diminishing returns. (Adding *trees* is not like adding *depth*: it steadies the
vote, it doesn't drive memorising.)

In [ ]:
ns = [1, 5, 10, 25, 100]
accs = []
for n in ns:
    f = RandomForestClassifier(n_estimators=n, random_state=0).fit(X_train, y_train)
    accs.append(f.score(X_test, y_test))
    print(f'{n:3d} trees: accuracy {accs[-1]:.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ns, accs, 'o-')
ax.set_xlabel('number of trees (n_estimators)'); ax.set_ylabel('test accuracy')
ax.set_title('More experts help, then plateau'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Step 6 · ✏️ What did we give up?

In Session 14 a single tree was our poster child for **interpretability** — you could
read it aloud as if-then rules and show it to a customer. A forest of 100 deep trees
cannot be read that way.

**✏️ Your reflection (2–3 sentences):** what did the forest cost us compared with one
readable tree, and can you imagine a situation where that cost would make you choose the
single tree anyway?

*(Write your answer here, replacing this line.)*

## Wrap-up

- A **random forest** = many trees voting, made different by random **rows** and **features**.
- The vote **cancels each tree's private mistakes** — here, 2 fewer false alarms, same catches.
- It **resists overfitting** where a single deep tree is brittle...
- ...but **costs interpretability** — you can't read 100 trees aloud.

**Next (S18):** forest vs single tree vs logistic — one problem, one metrics table, and a
real decision about which to ship (with a surprise).